# FraudShield Canton Backend API Verification

Run this notebook after deploying FraudShield. It verifies Canton through backend APIs only: readiness, Canton projection collections, seeded party mappings, and optional transaction flow checks.


In [ ]:
import json
import os
import time
from urllib.parse import urljoin

import requests

BASE_URL = os.getenv('FRAUDSHIELD_BASE_URL', 'http://localhost:8080').rstrip('/')
TIMEOUT = float(os.getenv('FRAUDSHIELD_TIMEOUT_SECONDS', '10'))
print(f'Using backend: {BASE_URL}')


In [ ]:
def request(method, path, **kwargs):
    url = urljoin(BASE_URL + '/', path.lstrip('/'))
    response = requests.request(method, url, timeout=TIMEOUT, **kwargs)
    try:
        payload = response.json()
    except ValueError:
        payload = response.text
    print(f'{method} {path} -> {response.status_code}')
    print(json.dumps(payload, indent=2, default=str) if not isinstance(payload, str) else payload)
    return response, payload

def assert_status(response, expected, message):
    assert response.status_code in expected, f'{message}: expected {expected}, got {response.status_code}'


## 1. Health and readiness
`/ready` should include a `canton` object. If Canton is enabled and all endpoints are reachable, the readiness status should be `READY`.


In [ ]:
resp, health = request('GET', '/health')
assert_status(resp, {200}, 'health should be up')

resp, ready = request('GET', '/ready')
assert_status(resp, {200, 503}, 'ready endpoint should respond')
assert 'canton' in ready, 'Expected /ready response to include canton readiness details'
print('Canton readiness status:', ready['canton'].get('status'))


## 2. Canton status and Mongo projection collections
This checks the helper API that summarizes Canton readiness plus required MongoDB projection collections. Counts can be zero immediately after deployment; they should not be `MISSING`.


In [ ]:
resp, canton_status = request('GET', '/api/canton/status')
assert_status(resp, {200}, 'canton status should respond')
missing = [name for name, count in canton_status.get('collections', {}).items() if count == 'MISSING']
assert not missing, f'Missing Canton projection collections: {missing}'
print('All Canton projection collections exist.')


## 3. Existing users still work and expose Canton mapping metadata
Verifies U001-U007 and ADMIN are still returned by the existing users API and include bank/participant/Canton fields.


In [ ]:
expected_users = {'U001','U002','U003','U004','U005','U006','U007','ADMIN'}
resp, users = request('GET', '/api/users/all')
assert_status(resp, {200}, 'users API should respond')
by_id = {user['id']: user for user in users}
missing_users = sorted(expected_users - set(by_id))
assert not missing_users, f'Missing seeded users: {missing_users}'
for user_id in sorted(expected_users):
    user = by_id[user_id]
    for field in ('bankId', 'participantId', 'cantonPartyId', 'cantonRole'):
        assert user.get(field), f'{user_id} is missing {field}'
print('All expected users have Canton metadata.')


## 4. Canton party mapping endpoints
Verifies the dedicated mapping read APIs.


In [ ]:
resp, mappings = request('GET', '/api/canton/party-mappings')
assert_status(resp, {200}, 'party mappings API should respond')
mapping_ids = {mapping['appUserId'] for mapping in mappings}
assert expected_users.issubset(mapping_ids), f'Missing Canton party mappings: {sorted(expected_users - mapping_ids)}'

for user_id in sorted(expected_users):
    resp, mapping = request('GET', f'/api/canton/party-mappings/{user_id}')
    assert_status(resp, {200}, f'{user_id} mapping should exist')
    assert mapping['appUserId'] == user_id
print('All expected Canton party mappings are readable.')


## 5. Optional transaction smoke test
Set `RUN_TXN_SMOKE=true` before starting Jupyter if you want the notebook to initiate a transaction through the existing backend API. Adjust the payload if your deployment has stricter fraud-rule thresholds.


In [ ]:
RUN_TXN_SMOKE = os.getenv('RUN_TXN_SMOKE', 'false').lower() == 'true'
txn_payload = {
    'fromUserId': os.getenv('SMOKE_FROM_USER', 'U001'),
    'toUserId': os.getenv('SMOKE_TO_USER', 'U003'),
    'amount': float(os.getenv('SMOKE_AMOUNT', '25')),
    'transactionType': os.getenv('SMOKE_TRANSACTION_TYPE', 'DOMESTIC'),
    'bypassSelfLimits': os.getenv('SMOKE_BYPASS_SELF_LIMITS', 'false').lower() == 'true'
}
if RUN_TXN_SMOKE:
    resp, txn = request('POST', '/api/txn/initiate', json=txn_payload)
    assert_status(resp, {200}, 'transaction initiation should respond')
    txn_id = txn.get('txnId') or txn.get('id')
    print('Created transaction:', txn_id)
    request('GET', '/api/mempool/status')
    if txn_id:
        request('GET', f'/api/canton/contract-refs/{txn_id}')
else:
    print('Skipping transaction smoke test. Set RUN_TXN_SMOKE=true to enable it.')


## Result
If all assertions above pass, your deployed backend can verify Canton readiness, projection collections, and existing-user Canton mappings through backend APIs.
